# 텍스트마이닝 A to Z: 실습 중심 수업

학습 목표
- 텍스트 데이터를 수집·정제·특징화하여 분류/요약/주제추출까지 전체 파이프라인을 경험한다.
- 불용어 처리, n-그램, TF-IDF, WordCloud, 간단 감성분석, LDA 토픽모델링을 실습한다.

구성
1) 데이터 준비와 전처리(정규화, 토큰화, 불용어)
2) 빈도 분석과 시각화(상위 단어, n-그램, 워드클라우드)
3) 벡터화(TF, TF-IDF)
4) 간단 감성분석(어휘 기반)
5) 토픽모델링(LDA)
6) 확장 아이디어(문서 분류, 요약)

수행 순서
- 셀 2부터 순서대로 실행하세요. (각 셀은 독립적으로 동작하도록 구성됨)

In [ ]:
# 1) 필수 라이브러리 설치(필요 시) & 임포트
# - 로컬/Colab 환경에 따라 설치는 건너뛸 수 있습니다.
# - nltk 최초 실행 시 불용어 사전 다운로드가 필요합니다.

import sys, subprocess

def pip_install(pkg):
    try:
        __import__(pkg.split('==')[0])
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

# 필요시 주석 해제하여 설치
# for p in [
#     'pandas', 'numpy', 'matplotlib', 'seaborn', 'wordcloud',
#     'scikit-learn', 'nltk', 'konlpy', 'gensim'
# ]:
#     pip_install(p)

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA
import nltk

# nltk 불용어(영어) 다운로드 (최초 1회)
try:
    nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

print('✅ 라이브러리 준비 완료')

In [ ]:
# 2) 예시 데이터 구성 및 전처리 함수 정의
sample_docs = [
    "LG전자 반려동물 케어 제품이 좋다. 배송도 빠르고 포장도 깔끔했다!",
    "서비스가 아쉬웠지만, 제품 품질은 만족. 다시 살 의향 있음.",
    "가격이 너무 비싸다... 다른 대안을 찾아보는 중",
    "Design is sleek and performance is great. Highly recommend!",
    "The packaging was damaged but customer support resolved it quickly.",
]

en_stop = set(nltk.corpus.stopwords.words('english'))
kr_stop = set(['그리고','그런데','하지만','그러나','또한','또는','에서','으로','하다','했다','하는','있다'])

# 한/영 단순 정규화 + 토큰화
def simple_clean_tokenize(text: str):
    # 소문자, 한글-영문-숫자만 남기기
    text = text.lower()
    text = re.sub(r"[^0-9a-zA-Zㄱ-ㅎ가-힣\s]", " ", text)
    tokens = text.split()
    return tokens

# 불용어 제거
def remove_stopwords(tokens):
    cleaned = []
    for t in tokens:
        if re.fullmatch(r"[a-z]+", t):  # 영어
            if t not in en_stop and len(t) > 1:
                cleaned.append(t)
        elif re.fullmatch(r"[0-9]+", t):
            continue
        else:  # 한글 토큰 가정
            if t not in kr_stop and len(t) > 1:
                cleaned.append(t)
    return cleaned

# 파이프라인 적용
def preprocess_docs(docs):
    processed = []
    for d in docs:
        toks = simple_clean_tokenize(d)
        toks = remove_stopwords(toks)
        processed.append(" ".join(toks))
    return processed

processed_docs = preprocess_docs(sample_docs)

pd.DataFrame({
    'original': sample_docs,
    'processed': processed_docs
})

In [ ]:
# 3) 단어 빈도 수 분석 및 시각화
from collections import Counter

all_tokens = " ".join(processed_docs).split()
counts = Counter(all_tokens)

top_n = counts.most_common(15)

df_top = pd.DataFrame(top_n, columns=['word','count'])
plt.figure(figsize=(8,4))
sns.barplot(data=df_top, x='count', y='word', palette='Blues_r')
plt.title('Top Terms (Count)')
plt.tight_layout()
plt.show()

df_top

In [ ]:
# 4) N-그램(예: Bigrams) 분석
from sklearn.feature_extraction.text import CountVectorizer

bigram_vectorizer = CountVectorizer(ngram_range=(2,2))
X_bi = bigram_vectorizer.fit_transform(processed_docs)
bi_counts = np.asarray(X_bi.sum(axis=0)).ravel()
bi_terms = np.array(list(bigram_vectorizer.get_feature_names_out()))

bi_freq = pd.DataFrame({'bigram': bi_terms, 'count': bi_counts})
bi_top = bi_freq.sort_values('count', ascending=False).head(15)

plt.figure(figsize=(8,4))
sns.barplot(data=bi_top, x='count', y='bigram', palette='Greens_r')
plt.title('Top Bigrams')
plt.tight_layout()
plt.show()

bi_top

In [ ]:
# 5) TF-IDF 벡터화
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(min_df=1)
X_tfidf = tfidf.fit_transform(processed_docs)

# 상위 TF-IDF 단어 확인(문서 평균 기준)
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()
terms = np.array(tfidf.get_feature_names_out())

tfidf_df = pd.DataFrame({'term': terms, 'score': mean_tfidf})
show_tfidf = tfidf_df.sort_values('score', ascending=False).head(15)

plt.figure(figsize=(8,4))
sns.barplot(data=show_tfidf, x='score', y='term', palette='Oranges_r')
plt.title('Top Terms (TF-IDF mean)')
plt.tight_layout()
plt.show()

show_tfidf

In [ ]:
# 6) 워드클라우드 생성
text_for_wc = " ".join(processed_docs)

wc = WordCloud(
    width=800, height=500,
    background_color='white',
    colormap='tab20c',
    font_path=None # 한글 글꼴 경로 필요 시 설정 (예: 'C:/Windows/Fonts/malgun.ttf')
).generate(text_for_wc)

plt.figure(figsize=(10,6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud')
plt.show()

In [ ]:
# 7) 간단 감성분석(어휘 기반)
pos_words = set(['good','great','recommend','만족','좋다','빠르고','깔끔'])
neg_words = set(['bad','expensive','비싸다','아쉽','damage','damaged'])

scores = []
for doc in processed_docs:
    toks = doc.split()
    pos = sum(1 for t in toks if t in pos_words)
    neg = sum(1 for t in toks if t in neg_words)
    scores.append(pos - neg)

sent_df = pd.DataFrame({'doc': sample_docs, 'score': scores})
sent_df['label'] = np.where(sent_df['score']>0, 'positive', np.where(sent_df['score']<0, 'negative', 'neutral'))

plt.figure(figsize=(6,3))
sns.countplot(data=sent_df, x='label', palette='Set2')
plt.title('Sentiment (lexicon-based)')
plt.show()

sent_df

In [ ]:
# 8) LDA 토픽모델링 (간단 예)
# 주의: 토픽 모델링은 문서 수가 충분히 많을 때 더 유의미합니다.

# 영문 stopword만 제거하여 LDA용으로 간단 구성
lda_vec = CountVectorizer(stop_words='english', min_df=1)
X_lda = lda_vec.fit_transform(processed_docs)

lda = LDA(n_components=2, random_state=42)
lda.fit(X_lda)

terms = np.array(lda_vec.get_feature_names_out())

def show_topics(model, feature_names, n_top_words=8):
    topics = []
    for idx, comp in enumerate(model.components_):
        top_idx = comp.argsort()[::-1][:n_top_words]
        topics.append((idx, list(feature_names[top_idx])))
    return topics

show_topics(lda, terms)